# CareFlow: Clinical Pathway Process Mining
## Member 2 Deliverable: Data Cleaning, Quality Assessment & Pipeline

This notebook performs exploratory data analysis, data quality audits, and cleaning on the raw hospital outpatient (OPD) dataset.

In [ ]:
import pandas as pd
import numpy as np

# 1. Load raw dataset
df_raw = pd.read_csv('data/uncleaned_HospitalOPD.csv')
print(f'Raw Dataset Shape: {df_raw.shape}')
df_raw.head()

### Step 1: Structural Audit & Dropping Ghost Columns
- Detecting double comma in CSV headers producing `Unnamed: 2` with 100% missing values.

In [ ]:
df_clean = df_raw.copy()
if 'Unnamed: 2' in df_clean.columns:
    df_clean.drop(columns=['Unnamed: 2'], inplace=True)
    print('Dropped empty ghost column: Unnamed: 2')

### Step 2: String Trimming & Formatting
- Standardizing doctor names and removing leading/trailing spaces.

In [ ]:
str_cols = ['prescription_doctor_name', 'lab_name', 'imaging_name', 'medicine_name', 'flow_name']
for col in str_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan, '': np.nan})

df_clean['prescription_doctor_name'] = df_clean['prescription_doctor_name'].apply(
    lambda x: ' '.join(x.split()) if pd.notna(x) else x
)

### Step 3: Timestamp Parsing & Chronological Validation
- Coerce corrupt/truncated time formats in `lab_end_datetime` to `NaT`.

In [ ]:
datetime_cols = [
    'appointment_start_datetime', 'prescription_start_datetime', 'prescription_end_datetime',
    'lab_start_datetime', 'lab_end_datetime', 'imaging_start_datetime', 'imaging_end_datetime',
    'pharmacy_start_datetime', 'pharmacy_end_datetime'
]
for col in datetime_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
print('All 9 datetime columns successfully standardized.')

### Step 4: Numeric Foreign Key Casting
- Cast float-formatted IDs to nullable Int64.

In [ ]:
id_cols = ['lab_id', 'imaging_request_id', 'imaging_result_id', 'pharmacy_sale_line_id']
for col in id_cols:
    df_clean[col] = df_clean[col].astype('Int64')

### Step 5: Duplicate Detection & Primary Key Re-indexing

In [ ]:
feature_cols = [c for c in df_clean.columns if c != 'record_no']
dups = df_clean.duplicated(subset=feature_cols).sum()
print(f'Duplicate records found: {dups}')
df_clean.drop_duplicates(subset=feature_cols, inplace=True)
df_clean.reset_index(drop=True, inplace=True)
df_clean['record_no'] = range(1, len(df_clean) + 1)

# Export to clean storage
for col in datetime_cols:
    df_clean[col] = df_clean[col].dt.strftime('%Y-%m-%d %H:%M:%S')
df_clean.to_csv('data/cleaned_HospitalOPD.csv', index=False)
print(f'Cleaned dataset saved: {df_clean.shape}')